In [ ]:
#w4
#Import Library
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score
import json
import os

#กําหนด CONFIG
TRAIN_CSV = r"F:\for-learning\Data\synthetic_plant_train.csv"               #ระบุไฟล์ข้อมูล Train      
MODEL_OUT = r"F:\for-learning\Data\xgb_plant_model.json"                    #ระบุ path
META_OUT  = r"F:\for-learning\Data\xgb_plant_model_meta.json"               #ระบุ path

FEATURES = ["temp_c", "humidity_pct", "lux", "vpd_kpa"]                     #ตัวแปรinput
TARGET = "y"                                                                #คลาสที่ต้องทำนาย
LABELS = [0, 1, 2]                                                          #มีทั้งหมดคลาส

#โหลดและเตรียมขอมูล
#NOTE: Load TRAIN data only
train_df = pd.read_csv(TRAIN_CSV)                                           #โหลด CSV
train_df = train_df.dropna(subset=FEATURES + [TARGET])                      #ลบแถวทีมีค่า missing ใน feature หรือ target

X_train = train_df[FEATURES].values                                         #แยก X (features) และ y (label)
y_train = train_df[TARGET].astype(int).values                               #แปลง y เป็น int (จําเป็นสําหรับ class)

print("Train samples:", len(y_train))                                       #ดูจํานวนข้อมูล
print("Train class distribution:")                                          #ตรวจสอบ class imbalance
print(train_df[TARGET].value_counts())

#สร้างและเทรน XGBoost
#NOTE: Train model
model = XGBClassifier(
    n_estimators=200,                                                       # จำนวนต้นไม้ที่จะสร้าง (ยิ่งเยอะยิ่งละเอียด แต่อาจจะ Overfit)
    max_depth=4,                                                            # ความลึกสูงสุดของต้นไม้         
    learning_rate=0.05,                                                     # อัตราการเรียนรู้ (ก้าวทีละน้อยเพื่อให้แม่นยำ)
    subsample=0.8,                                                          # สุ่มใช้ข้อมูล 80% ในการสร้างต้นไม้แต่ละต้น (ลดการ Overfit)
    colsample_bytree=0.8,                                                   # สุ่มเลือกฟีเจอร์ 80% ต่อหนึ่งต้นไม้
    objective="multi:softprob",                                             # ระบุว่าทำ Multi-class classification (ได้ผลลัพธ์เป็นความน่าจะเป็น)
    num_class=3,                                                            # ระบุจำนวนคลาส (0, 1, 2)
    tree_method="hist",                                                     # ใช้วิธี Histogram-based เพื่อความรวดเร็วในการเทรน  
    eval_metric="mlogloss",                                                 # ใช้ Log Loss เป็นตัววัดความผิดพลาดขณะเทรน
    random_state=42,                                                        # ล็อกค่าสุ่มเพื่อให้รันกี่ครั้งก็ได้ผลเหมือนเดิม
)

model.fit(X_train, y_train)                                                 # เริ่มกระบวนการเรียนรู้ (Training) จากข้อมูล X และ y

#ประเมินผลบน Train
train_pred = model.predict(X_train)                                         #ให้โมเดลลองทำนายข้อมูล
train_f1 = f1_score(y_train, train_pred, average="macro", labels=LABELS)    #คำนวณค่า F1-Score ให้ความสําคัญทุก class เท่ากัน
train_acc = accuracy_score(y_train, train_pred)                             #คำนวณความแม่นยำโดยรวม (Accuracy)

print(f"\nTRAIN Accuracy : {train_acc:.4f}")
print(f"TRAIN Macro-F1 : {train_f1:.4f}")

#เซฟ Model
#NOTE: Save model + metadata
os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)                      # ตรวจสอบและสร้าง Folder ถ้ายังไม่มี
model.save_model(MODEL_OUT)                                                 # บันทึกตัวโมเดลเป็นไฟล์ .json

#เซฟ Metadata
meta = {
    "features": FEATURES,                                                   #โมเดลใช้ feature อะไร
    "labels": LABELS,                                                       #label อะไร
    "train_samples": int(len(y_train)),                                     #เทรนด้วยพารามิเตอร์อะไร
    "train_macro_f1": float(train_f1),
    "train_accuracy": float(train_acc),                                     
    "model_params": model.get_params(),                                     #เก็บพารามิเตอร์ที่ใช้เทรนโมเดล
}

# บันทึก Metadata ลงไฟล์ .json
with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"\n✅ Model saved to: {MODEL_OUT}")                                  #แสดงข้อความยืนยันตําแหน่งไฟล์โมเดลทีบันทึกแล้ว
print(f"🧾 Metadata saved to: {META_OUT}")                                  #แสดงข้อความยืนยันตําแหน่งไฟล์ metadata ทีบันทึกแล้ว

Train samples: 1958
Train class distribution:
y
1.0    1055
0.0     838
2.0      65
Name: count, dtype: int64

TRAIN Accuracy : 0.9990
TRAIN Macro-F1 : 0.9993

✅ Model saved to: F:\for-learning\Data\xgb_plant_model.json
🧾 Metadata saved to: F:\for-learning\Data\xgb_plant_model_meta.json
